# TacticsGPT Phase 1: Colab Training Notebook

This notebook builds a football tactics GPT in stages: corpus cleaning, tokenizer training, base pretraining, SFT LoRA fine-tuning, RL LoRA tuning, and generation/evaluation checks.

## Quick Run Map

1. Install dependencies.
2. Mount Google Drive and enter the project folder.
3. Add or clean the corpus only if `data/tactics_corpus.txt` is missing or intentionally being rebuilt.
4. Train the tokenizer only if `checkpoints/tokenizer/tokenizer.json` is missing or intentionally being rebuilt.
5. Run the stage you want to continue.

## Resume Disclaimer

Keep `START_FROM_SCRATCH = False` and keep `PROJECT_DIR` pointed at the same Google Drive folder when you want to continue from old weights.

- Pretraining resumes from the latest `checkpoints/pretrain/checkpoint_step_*.pt` unless `--no_resume` is used.
- SFT loads the pretrained base checkpoint and resumes the latest `checkpoints/sft/sft_step_*.pt`.
- RL loads the pretrained base plus the latest SFT adapter, then resumes the latest `rl_step_*.pt` in the chosen RL output folder.
- Evaluation cells only load existing weights and do not train.

Only set `START_FROM_SCRATCH = True`, change `PROJECT_DIR`, or change a training `--out_dir` when you intentionally want a fresh run.


## 0. Install Dependencies

Run once per Colab runtime. These packages are needed by the tokenizer, PyTorch model, progress bars, and data utilities.


In [ ]:
%pip install -q tokenizers tqdm numpy torch

## 1. Mount Drive And Project Folder

Run at the start of every Colab session. Keep `START_FROM_SCRATCH = False` to preserve existing Drive checkpoints and continue from old weights.


In [ ]:
from pathlib import Path
import os
import torch

from google.colab import drive

# Google Drive is always used so checkpoints survive Colab runtime resets.
drive.mount("/content/drive", force_remount=True)
PROJECT_DIR = Path("/content/drive/MyDrive/TacticsGPT_Phase1_Full_Pretrain")

# Keep this False for normal use. Because this notebook uses a new project folder,
# the first run starts clean, and later Colab restarts can resume from Drive.
START_FROM_SCRATCH = False

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

if START_FROM_SCRATCH:
    for target in [
        Path("checkpoints/pretrain"),
        Path("checkpoints/tokenizer"),
        Path("data/tactics_corpus.txt"),
    ]:
        if target.is_dir():
            import shutil
            shutil.rmtree(target)
        elif target.exists():
            target.unlink()
    print("Fresh run enabled: removed old generated tokenizer, cleaned corpus, and pretrain checkpoints.")
else:
    print("Resume mode: existing Drive corpus, tokenizer, and checkpoints will be kept.")

for folder in [
    "data/tokenizer",
    "checkpoints/pretrain",
    "checkpoints/tokenizer",
    "src",
]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable Runtime > Change runtime type > GPU before full pretraining.")


## 2. Add And Clean The Training Corpus

Run this when `data/tactics_corpus.txt` is missing or when you intentionally want to rebuild the corpus from a new raw text dump.

Skip this while simply continuing old weights if your cleaned corpus is already in Drive.

The cleaned output is saved to `data/tactics_corpus.txt` as many training documents separated by blank lines. Long sections are chunked into smaller documents so the model does not treat a huge mixed tail as one single article.


In [ ]:
from pathlib import Path
import errno
import re
import unicodedata

raw_path = Path("data/raw_tactical_match_analysis.txt")
corpus_path = Path("data/tactics_corpus.txt")


def is_transport_disconnect(exc: OSError) -> bool:
    return (
        getattr(exc, "errno", None) == errno.ENOTCONN
        or "Transport endpoint is not connected" in str(exc)
    )


def raise_storage_help(path: Path, exc: OSError) -> None:
    raise RuntimeError(
        f"Colab cannot access {path!s}. This usually means the current project folder "
        "is on a disconnected Google Drive mount.\n\n"
        "Fix options:\n"
        "1. Remount Drive with: drive.mount('/content/drive', force_remount=True), then rerun from the setup cell.\n"
        "2. If the Runtime is stale, use Runtime > Restart session before rerunning."
    ) from exc


def safe_exists(path: Path) -> bool:
    try:
        return path.exists()
    except OSError as exc:
        if is_transport_disconnect(exc):
            raise_storage_help(path, exc)
        raise


def safe_size(path: Path) -> int:
    try:
        return path.stat().st_size
    except OSError as exc:
        if is_transport_disconnect(exc):
            raise_storage_help(path, exc)
        raise


def verify_data_storage() -> None:
    try:
        Path("data").mkdir(parents=True, exist_ok=True)
        probe = Path("data/.storage_probe")
        probe.write_text("ok", encoding="utf-8")
        probe.unlink(missing_ok=True)
    except OSError as exc:
        if is_transport_disconnect(exc):
            raise_storage_help(Path("data"), exc)
        raise


def maybe_fix_mojibake(text: str) -> str:
    markers = ("â€™", "â€œ", "â€", "â€”", "â€“", "Ã©")
    if sum(text.count(marker) for marker in markers) < 5:
        return text
    try:
        return text.encode("cp1252").decode("utf-8")
    except UnicodeError:
        return text


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", maybe_fix_mojibake(text))
    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u00a0": " ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def is_wrapper_line(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    if stripped == "TACTICAL MATCH ANALYSIS ARTICLES":
        return True
    if stripped.startswith("Generated by "):
        return True
    if re.fullmatch(r"=+", stripped):
        return True
    if re.fullmatch(r"ARTICLE\s+#\d+", stripped, flags=re.IGNORECASE):
        return True
    return False


def is_document_boundary(line: str) -> bool:
    stripped = line.strip()
    return bool(re.fullmatch(r"ARTICLE\s+#\d+", stripped, flags=re.IGNORECASE))


def paragraph_chunks(paragraphs: list[str], target_chars: int = 9000, max_chars: int = 14000) -> list[str]:
    docs = []
    current = []
    current_len = 0

    for paragraph in paragraphs:
        paragraph = " ".join(paragraph.split())
        if not paragraph:
            continue

        if current and current_len + len(paragraph) > target_chars:
            docs.append("\n\n".join(current))
            current = []
            current_len = 0

        current.append(paragraph)
        current_len += len(paragraph)

        if current_len >= max_chars:
            docs.append("\n\n".join(current))
            current = []
            current_len = 0

    if current:
        docs.append("\n\n".join(current))

    return docs


def extract_training_documents(raw_text: str, min_chars: int = 400) -> list[str]:
    text = normalize_text(raw_text).replace("\r\n", "\n").replace("\r", "\n")
    sections: list[list[str]] = []
    current_lines: list[str] = []

    for line in text.splitlines():
        if is_document_boundary(line) and current_lines:
            sections.append(current_lines)
            current_lines = []
            continue
        if is_wrapper_line(line):
            continue
        current_lines.append(line.rstrip())

    if current_lines:
        sections.append(current_lines)

    documents = []
    for section in sections:
        section_text = "\n".join(section).strip()
        if not section_text:
            continue
        paragraphs = [part.strip() for part in re.split(r"\n\s*\n+", section_text) if part.strip()]
        for doc in paragraph_chunks(paragraphs):
            if len(doc) >= min_chars:
                documents.append(doc)

    return documents


def build_clean_corpus(raw_file: Path, clean_file: Path) -> list[str]:
    raw_text = raw_file.read_text(encoding="utf-8", errors="replace")
    documents = extract_training_documents(raw_text)
    if not documents:
        raise ValueError("No training documents were found. Check the uploaded text file format.")
    clean_file.write_text("\n\n\n".join(documents) + "\n", encoding="utf-8")
    return documents


verify_data_storage()
raw_exists = safe_exists(raw_path)
corpus_exists = safe_exists(corpus_path)

if not raw_exists and not corpus_exists:
    try:
        from google.colab import files

        print("Upload tactical_match_analysis_5mb.txt or a similar tactics .txt dump.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")

        first_name = next(iter(uploaded))
        raw_path.write_bytes(uploaded[first_name])
        raw_exists = True
        print(f"Saved uploaded file as {raw_path}")
    except Exception as exc:
        print("Could not upload automatically:", exc)
        print("Place your raw text dump at data/raw_tactical_match_analysis.txt before continuing.")

if raw_exists:
    documents = build_clean_corpus(raw_path, corpus_path)
    print(f"Cleaned {len(documents):,} training documents/sections into {corpus_path}")
elif corpus_exists and safe_size(corpus_path) > 0:
    documents = [part.strip() for part in re.split(r"\n{3,}", corpus_path.read_text(encoding="utf-8", errors="replace")) if part.strip()]
    print(f"Using existing cleaned corpus with {len(documents):,} training documents/sections.")
else:
    raise FileNotFoundError("Missing data/raw_tactical_match_analysis.txt or data/tactics_corpus.txt")

print(f"Clean corpus size: {safe_size(corpus_path):,} bytes")
print("\nFirst cleaned document preview:\n")
print(documents[0][:1200])

## 3. Write Project Source Files

These cells create the Python scripts used by the notebook. Run them after setup in a fresh Colab session, or after editing any script logic.


### 3.1 Shared Helpers - `src/utils.py`

Creates device selection, seed setup, folder creation, and checkpoint helper functions used by the training scripts.


In [ ]:
%%writefile src/utils.py
import glob
import os
import random
import re
from pathlib import Path

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def ensure_dirs(*paths: str) -> None:
    for path in paths:
        Path(path).mkdir(parents=True, exist_ok=True)


def checkpoint_step(path: str) -> int:
    match = re.search(r"checkpoint_step_(\d+)\.pt$", os.path.basename(path))
    return int(match.group(1)) if match else -1


def latest_checkpoint(out_dir: str) -> str | None:
    checkpoints = glob.glob(os.path.join(out_dir, "checkpoint_step_*.pt"))
    if not checkpoints:
        return None
    return max(checkpoints, key=checkpoint_step)


def save_checkpoint(
    path: str,
    model,
    optimizer,
    step: int,
    epoch: int,
    tokenizer_path: str,
    config: dict,
    best_val_loss: float | None = None,
) -> None:
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "epoch": epoch,
        "tokenizer_path": tokenizer_path,
        "config": config,
    }
    if best_val_loss is not None:
        payload["best_val_loss"] = best_val_loss
    torch.save(payload, path)

### 3.2 Corpus Builder - `src/build_corpus.py`

Writes the reusable corpus-cleaning script for raw tactical text dumps.


In [ ]:
%%writefile src/build_corpus.py
import argparse
from pathlib import Path
import re
import unicodedata


def maybe_fix_mojibake(text: str) -> str:
    markers = ("â€™", "â€œ", "â€", "â€”", "â€“", "Ã©")
    if sum(text.count(marker) for marker in markers) < 5:
        return text
    try:
        return text.encode("cp1252").decode("utf-8")
    except UnicodeError:
        return text


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", maybe_fix_mojibake(text))
    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u00a0": " ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def is_wrapper_line(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    if stripped == "TACTICAL MATCH ANALYSIS ARTICLES":
        return True
    if stripped.startswith("Generated by "):
        return True
    if re.fullmatch(r"=+", stripped):
        return True
    if re.fullmatch(r"ARTICLE\s+#\d+", stripped, flags=re.IGNORECASE):
        return True
    return False


def is_document_boundary(line: str) -> bool:
    stripped = line.strip()
    return bool(re.fullmatch(r"ARTICLE\s+#\d+", stripped, flags=re.IGNORECASE))


def paragraph_chunks(paragraphs: list[str], target_chars: int = 9000, max_chars: int = 14000) -> list[str]:
    docs = []
    current = []
    current_len = 0

    for paragraph in paragraphs:
        paragraph = " ".join(paragraph.split())
        if not paragraph:
            continue

        if current and current_len + len(paragraph) > target_chars:
            docs.append("\n\n".join(current))
            current = []
            current_len = 0

        current.append(paragraph)
        current_len += len(paragraph)

        if current_len >= max_chars:
            docs.append("\n\n".join(current))
            current = []
            current_len = 0

    if current:
        docs.append("\n\n".join(current))

    return docs


def extract_training_documents(raw_text: str, min_chars: int = 400) -> list[str]:
    text = normalize_text(raw_text).replace("\r\n", "\n").replace("\r", "\n")
    sections: list[list[str]] = []
    current_lines: list[str] = []

    for line in text.splitlines():
        if is_document_boundary(line) and current_lines:
            sections.append(current_lines)
            current_lines = []
            continue
        if is_wrapper_line(line):
            continue
        current_lines.append(line.rstrip())

    if current_lines:
        sections.append(current_lines)

    documents = []
    for section in sections:
        section_text = "\n".join(section).strip()
        if not section_text:
            continue
        paragraphs = [part.strip() for part in re.split(r"\n\s*\n+", section_text) if part.strip()]
        for doc in paragraph_chunks(paragraphs):
            if len(doc) >= min_chars:
                documents.append(doc)

    return documents


def build_clean_corpus(raw_file: str, clean_file: str, min_chars: int = 400) -> None:
    raw_path = Path(raw_file)
    out_path = Path(clean_file)
    if not raw_path.exists() or raw_path.stat().st_size == 0:
        raise FileNotFoundError(f"Raw corpus not found or empty: {raw_path}")

    documents = extract_training_documents(raw_path.read_text(encoding="utf-8", errors="replace"), min_chars)
    if not documents:
        raise ValueError("No training documents were found. Check the source text format.")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n\n\n".join(documents) + "\n", encoding="utf-8")
    print(f"Wrote {len(documents):,} cleaned training documents/sections to {out_path}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--raw", default="data/raw_tactical_match_analysis.txt")
    parser.add_argument("--out", default="data/tactics_corpus.txt")
    parser.add_argument("--min_chars", type=int, default=400)
    args = parser.parse_args()
    build_clean_corpus(args.raw, args.out, args.min_chars)


if __name__ == "__main__":
    main()

### 3.3 Tokenizer Trainer - `src/tokenizer_train.py`

Writes the BPE tokenizer training script.


In [ ]:
%%writefile src/tokenizer_train.py
import argparse
from pathlib import Path

from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import BPE
from tokenizers.normalizers import Lowercase, NFKC, Sequence
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import BpeTrainer


SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]


def train_tokenizer(corpus_path: str, out_dir: str, vocab_size: int = 8000, lowercase: bool = False) -> None:
    corpus = Path(corpus_path)
    if not corpus.exists() or corpus.stat().st_size == 0:
        raise FileNotFoundError(f"Corpus not found or empty: {corpus}")

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    normalizers = [NFKC()]
    if lowercase:
        normalizers.append(Lowercase())
    tokenizer.normalizer = Sequence(normalizers)
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
    )
    tokenizer.train([str(corpus)], trainer)

    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")
    tokenizer.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
    )

    tokenizer.save(str(out / "tokenizer.json"))
    tokenizer.model.save(str(out))

    print("Saved tokenizer files:")
    print(" -", out / "tokenizer.json")
    print(" -", out / "vocab.json")
    print(" -", out / "merges.txt")
    print("Actual vocab size:", tokenizer.get_vocab_size())


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--corpus", default="data/tactics_corpus.txt")
    parser.add_argument("--out_dir", default="checkpoints/tokenizer")
    parser.add_argument("--vocab_size", type=int, default=8000)
    parser.add_argument("--lowercase", action="store_true")
    args = parser.parse_args()
    train_tokenizer(args.corpus, args.out_dir, args.vocab_size, args.lowercase)


if __name__ == "__main__":
    main()

### 3.4 Pretraining Dataset - `src/dataset.py`

Writes the article-aware next-token dataset used during base pretraining.


In [ ]:
%%writefile src/dataset.py
import re
from pathlib import Path

import torch
from torch.utils.data import Dataset
from tokenizers import Tokenizer


def split_article_documents(text: str) -> list[str]:
    docs = [part.strip() for part in re.split(r"\n{3,}", text) if part.strip()]
    return docs if docs else [text.strip()]


class TacticsDataset(Dataset):
    def __init__(
        self,
        corpus_path: str,
        tokenizer_path: str,
        context_length: int = 256,
        stride: int | None = None,
    ) -> None:
        corpus = Path(corpus_path)
        if not corpus.exists() or corpus.stat().st_size == 0:
            raise FileNotFoundError(f"Corpus not found or empty: {corpus}")

        self.context_length = context_length
        self.stride = stride or context_length
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.documents: list[list[int]] = []
        self.examples: list[tuple[int, int]] = []

        text = corpus.read_text(encoding="utf-8", errors="ignore")
        for document in split_article_documents(text):
            token_ids = self.tokenizer.encode(document).ids
            if len(token_ids) < context_length + 1:
                continue
            doc_index = len(self.documents)
            self.documents.append(token_ids)
            self.examples.extend(
                (doc_index, start)
                for start in range(0, len(token_ids) - context_length, self.stride)
            )

        self.num_documents = len(self.documents)
        self.num_tokens = sum(len(doc) for doc in self.documents)
        if not self.examples:
            raise ValueError(
                "No training windows were created. "
                f"Try reducing context_length below {context_length}, "
                "or check that data/tactics_corpus.txt contains full article bodies."
            )

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int):
        doc_index, start = self.examples[idx]
        tokens = self.documents[doc_index]
        end = start + self.context_length
        x = torch.tensor(tokens[start:end], dtype=torch.long)
        y = torch.tensor(tokens[start + 1 : end + 1], dtype=torch.long)
        return x, y

### 3.5 GPT Model - `src/model.py`

Writes the decoder-only GPT architecture used by pretraining, SFT, RL, and generation.


In [ ]:
%%writefile src/model.py
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 8000
    context_length: int = 256
    n_layers: int = 4
    n_heads: int = 4
    d_model: int = 256
    ffn_hidden: int = 1024
    dropout: float = 0.1


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.out = nn.Linear(config.d_model, config.d_model)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        mask = torch.tril(torch.ones(config.context_length, config.context_length))
        self.register_buffer("causal_mask", mask.view(1, 1, config.context_length, config.context_length))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, channels = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(channels, dim=2)

        q = q.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        scores = scores.masked_fill(self.causal_mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        y = weights @ v
        y = y.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        return self.resid_dropout(self.out(y))


class FeedForward(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.ffn_hidden),
            nn.GELU(),
            nn.Linear(config.ffn_hidden, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.ffn = FeedForward(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.context_length, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        batch, seq_len = idx.shape
        if seq_len > self.config.context_length:
            raise ValueError(f"Sequence length {seq_len} exceeds context length {self.config.context_length}")

        positions = torch.arange(0, seq_len, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int = 80,
        temperature: float = 0.8,
        top_k: int | None = 50,
        greedy: bool = False,
    ) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.context_length :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            if greedy:
                next_id = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / max(temperature, 1e-6)
                if top_k is not None and top_k > 0:
                    values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < values[:, [-1]]] = float("-inf")
                probs = F.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)

            idx = torch.cat([idx, next_id], dim=1)
        return idx

### 3.6 Pretraining Script - `src/train_pretrain.py`

Writes the base training loop. It saves step checkpoints, a best validation checkpoint, and resumes automatically unless `--no_resume` is used.


In [ ]:
%%writefile src/wandb_utils.py
import os


def add_wandb_args(parser, default_group="tacticsgpt"):
    parser.add_argument("--wandb_project", default="", help="Enable W&B logging by setting a project name.")
    parser.add_argument("--wandb_entity", default="", help="Optional W&B entity/team.")
    parser.add_argument("--wandb_run_name", default="", help="Optional W&B run name.")
    parser.add_argument("--wandb_group", default=default_group, help="Optional W&B group name.")
    parser.add_argument("--wandb_mode", default=os.environ.get("WANDB_MODE", "online"), choices=["online", "offline", "disabled"])
    parser.add_argument("--wandb_tags", default="", help="Comma-separated W&B tags.")


def init_wandb(args, stage, config):
    if not getattr(args, "wandb_project", ""):
        return None

    try:
        import wandb
    except ImportError as exc:
        raise RuntimeError("W&B logging requested. Install it with `pip install wandb`.") from exc

    tags = [tag.strip() for tag in getattr(args, "wandb_tags", "").split(",") if tag.strip()]
    return wandb.init(
        project=args.wandb_project,
        entity=args.wandb_entity or None,
        name=args.wandb_run_name or None,
        group=args.wandb_group or "tacticsgpt",
        job_type=stage,
        mode=args.wandb_mode,
        tags=tags,
        config=config,
    )


def wandb_log(run, data, step=None):
    if run is not None:
        run.log(data, step=step)


def wandb_finish(run):
    if run is not None:
        run.finish()


In [ ]:
%%writefile src/summarize_metrics.py
import argparse
import json
import math
from pathlib import Path


def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def last_with(rows, *keys):
    for row in reversed(rows):
        if all(key in row and row[key] is not None for key in keys):
            return row
    return None


def reward_improvement_pct(current_reward, baseline_reward):
    if baseline_reward is None or not math.isfinite(baseline_reward) or abs(baseline_reward) < 1e-8:
        return None
    return 100.0 * (current_reward - baseline_reward) / abs(baseline_reward)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pretrain_metrics", default="checkpoints/pretrain/metrics.jsonl")
    ap.add_argument("--sft_metrics", default="checkpoints/sft/metrics.jsonl")
    ap.add_argument("--rl_metrics", default="checkpoints/rl/metrics.jsonl")
    args = ap.parse_args()

    pretrain_rows = read_jsonl(args.pretrain_metrics)
    sft_rows = read_jsonl(args.sft_metrics)
    rl_rows = read_jsonl(args.rl_metrics)

    pretrain_eval = last_with(pretrain_rows, "val_loss", "val_perplexity")
    sft_eval = last_with(sft_rows, "eval_loss", "eval_perplexity")

    rl_eval_rows = [row for row in rl_rows if "eval_reward" in row and row["eval_reward"] is not None]
    rl_best = max(rl_eval_rows, key=lambda row: row["eval_reward"]) if rl_eval_rows else None
    rl_first = rl_eval_rows[0] if rl_eval_rows else None

    print("TacticsGPT metrics summary")
    print("=" * 28)

    if pretrain_eval:
        print(f"Pretrain final val loss: {pretrain_eval['val_loss']:.4f}")
        print(f"Pretrain final perplexity: {pretrain_eval['val_perplexity']:.2f}")
    else:
        print("Pretrain eval metrics: not found")

    if sft_eval:
        print(f"SFT final eval loss: {sft_eval['eval_loss']:.4f}")
        print(f"SFT final perplexity: {sft_eval['eval_perplexity']:.2f}")
    else:
        print("SFT eval metrics: not found")

    improvement = None
    if rl_best and rl_first:
        improvement = rl_best.get("reward_improvement_pct")
        if improvement is None:
            improvement = reward_improvement_pct(float(rl_best["eval_reward"]), float(rl_first["eval_reward"]))
        print(f"GRPO first eval reward: {float(rl_first['eval_reward']):.4f}")
        print(f"GRPO best eval reward: {float(rl_best['eval_reward']):.4f}")
        if improvement is not None:
            print(f"GRPO reward improvement: {improvement:.2f}%")
        else:
            print("GRPO reward improvement: unavailable because the first eval reward is zero")
    else:
        print("GRPO reward metrics: not found")

    if sft_eval and improvement is not None:
        print()
        print("Resume bullet:")
        print(
            "- Built TacticsGPT, a domain-specific football tactics LLM, with integrated W&B "
            "experiment tracking across pretraining, SFT, and GRPO; logged training loss, "
            f"perplexity, and reward curves, reaching {sft_eval['eval_loss']:.4f} eval loss, "
            f"{sft_eval['eval_perplexity']:.2f} perplexity, and {improvement:.1f}% GRPO reward improvement."
        )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_pretrain.py
import argparse
import json
import math
from dataclasses import asdict
from pathlib import Path

import torch
from tokenizers import Tokenizer
from torch.utils.data import DataLoader, Subset, random_split
from tqdm.auto import tqdm

from dataset import TacticsDataset
from model import GPT, GPTConfig
from utils import ensure_dirs, get_device, latest_checkpoint, save_checkpoint, set_seed
from wandb_utils import add_wandb_args, init_wandb, wandb_finish, wandb_log


def make_loaders(dataset, batch_size: int, val_fraction: float, num_workers: int, seed: int, device: str):
    val_size = max(1, int(len(dataset) * val_fraction)) if val_fraction > 0 and len(dataset) > 20 else 0
    train_size = len(dataset) - val_size
    generator = torch.Generator().manual_seed(seed)

    if val_size:
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)
    else:
        train_dataset = dataset
        val_dataset = None

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=num_workers,
            pin_memory=(device == "cuda"),
        )
    return train_loader, val_loader, train_size, val_size


def learning_rate(step: int, base_lr: float, warmup_steps: int, max_steps: int) -> float:
    if warmup_steps > 0 and step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    if max_steps <= warmup_steps:
        return base_lr
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    progress = min(1.0, max(0.0, progress))
    return 0.1 * base_lr + 0.9 * base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, device: str, max_batches: int = 50) -> float | None:
    if loader is None:
        return None
    model.eval()
    losses = []
    for batch_idx, (x, y) in enumerate(loader):
        if batch_idx >= max_batches:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / max(1, len(losses))


def append_jsonl(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--corpus", default="data/tactics_corpus.txt")
    parser.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    parser.add_argument("--out_dir", default="checkpoints/pretrain")
    parser.add_argument("--context_length", type=int, default=256)
    parser.add_argument("--n_layers", type=int, default=4)
    parser.add_argument("--n_heads", type=int, default=4)
    parser.add_argument("--d_model", type=int, default=256)
    parser.add_argument("--ffn_hidden", type=int, default=1024)
    parser.add_argument("--dropout", type=float, default=0.1)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--grad_accum_steps", type=int, default=2)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--warmup_steps", type=int, default=500)
    parser.add_argument("--stride", type=int, default=128)
    parser.add_argument("--save_every", type=int, default=250)
    parser.add_argument("--eval_every", type=int, default=250)
    parser.add_argument("--val_fraction", type=float, default=0.05)
    parser.add_argument("--eval_batches", type=int, default=50)
    parser.add_argument("--max_steps", type=int, default=0, help="0 means train for all epochs")
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--log_every", type=int, default=10)
    parser.add_argument("--no_resume", action="store_true")
    add_wandb_args(parser)
    args = parser.parse_args()

    set_seed(args.seed)
    ensure_dirs(args.out_dir)
    metrics_path = Path(args.out_dir) / "metrics.jsonl"
    device = get_device()
    use_amp = device == "cuda"
    print("Device:", device)
    print("AMP:", use_amp)

    tokenizer = Tokenizer.from_file(args.tokenizer)
    config = GPTConfig(
        vocab_size=tokenizer.get_vocab_size(),
        context_length=args.context_length,
        n_layers=args.n_layers,
        n_heads=args.n_heads,
        d_model=args.d_model,
        ffn_hidden=args.ffn_hidden,
        dropout=args.dropout,
    )

    dataset = TacticsDataset(args.corpus, args.tokenizer, args.context_length, args.stride)
    train_loader, val_loader, train_size, val_size = make_loaders(
        dataset, args.batch_size, args.val_fraction, args.num_workers, args.seed, device
    )
    steps_per_epoch = max(1, len(train_loader) // max(1, args.grad_accum_steps))
    planned_steps = args.max_steps if args.max_steps else steps_per_epoch * args.epochs

    print(f"Dataset training documents/sections: {dataset.num_documents:,}")
    print(f"Dataset tokens: {dataset.num_tokens:,}")
    print(f"Dataset windows: {len(dataset):,}")
    print(f"Train windows: {train_size:,}")
    print(f"Validation windows: {val_size:,}")
    print(f"Optimization steps planned: {planned_steps:,}")
    print(f"Effective batch size: {args.batch_size * args.grad_accum_steps:,}")

    wandb_run = init_wandb(
        args,
        "pretrain",
        {
            "stage": "pretrain",
            "args": vars(args),
            "model": asdict(config),
            "dataset_documents": dataset.num_documents,
            "dataset_tokens": dataset.num_tokens,
            "dataset_windows": len(dataset),
            "train_windows": train_size,
            "val_windows": val_size,
            "planned_steps": planned_steps,
        },
    )

    model = GPT(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.95), weight_decay=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    global_step = 0
    start_epoch = 0
    best_val_loss = float("inf")

    ckpt = None if args.no_resume else latest_checkpoint(args.out_dir)
    if ckpt:
        print("Resuming from:", ckpt)
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        global_step = int(state.get("step", 0))
        start_epoch = int(state.get("epoch", 0))
        best_val_loss = float(state.get("best_val_loss", best_val_loss))

    model.train()
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(start_epoch, args.epochs):
        progress = tqdm(train_loader, desc=f"epoch {epoch + 1}/{args.epochs}")
        for micro_step, (x, y) in enumerate(progress, start=1):
            lr = learning_rate(global_step, args.lr, args.warmup_steps, planned_steps)
            for group in optimizer.param_groups:
                group["lr"] = lr

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                _, loss = model(x, y)
                loss_to_backprop = loss / args.grad_accum_steps

            scaler.scale(loss_to_backprop).backward()

            if micro_step % args.grad_accum_steps != 0:
                progress.set_postfix(loss=f"{loss.item():.4f}", step=global_step, lr=f"{lr:.2e}")
                continue

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            progress.set_postfix(loss=f"{loss.item():.4f}", step=global_step, lr=f"{lr:.2e}")
            if args.log_every > 0 and global_step % args.log_every == 0:
                append_jsonl(metrics_path, {
                    "stage": "pretrain",
                    "step": global_step,
                    "epoch": epoch + 1,
                    "train_loss": float(loss.item()),
                    "lr": float(lr),
                })
                wandb_log(
                    wandb_run,
                    {
                        "pretrain/train_loss": float(loss.item()),
                        "pretrain/lr": float(lr),
                        "pretrain/epoch": epoch + 1,
                    },
                    step=global_step,
                )

            should_eval = val_loader is not None and args.eval_every > 0 and global_step % args.eval_every == 0
            if should_eval:
                val_loss = evaluate(model, val_loader, device, args.eval_batches)
                val_ppl = math.exp(min(val_loss, 20))
                print(f"\\nValidation loss at step {global_step}: {val_loss:.4f}")
                append_jsonl(metrics_path, {
                    "stage": "pretrain",
                    "step": global_step,
                    "epoch": epoch + 1,
                    "val_loss": float(val_loss),
                    "val_perplexity": float(val_ppl),
                    "best_val_loss": float(min(best_val_loss, val_loss)),
                })
                wandb_log(
                    wandb_run,
                    {
                        "pretrain/val_loss": float(val_loss),
                        "pretrain/val_perplexity": float(val_ppl),
                        "pretrain/best_val_loss": float(min(best_val_loss, val_loss)),
                    },
                    step=global_step,
                )
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_path = Path(args.out_dir) / "checkpoint_best.pt"
                    save_checkpoint(str(best_path), model, optimizer, global_step, epoch, args.tokenizer, asdict(config), best_val_loss)
                    print("Saved new best checkpoint:", best_path)

            if global_step % args.save_every == 0:
                path = Path(args.out_dir) / f"checkpoint_step_{global_step}.pt"
                save_checkpoint(str(path), model, optimizer, global_step, epoch, args.tokenizer, asdict(config), best_val_loss)
                print("\\nSaved", path)

            if args.max_steps and global_step >= args.max_steps:
                if val_loader is not None and (args.eval_every <= 0 or global_step % args.eval_every != 0):
                    val_loss = evaluate(model, val_loader, device, args.eval_batches)
                    val_ppl = math.exp(min(val_loss, 20))
                    append_jsonl(metrics_path, {
                        "stage": "pretrain",
                        "step": global_step,
                        "epoch": epoch + 1,
                        "val_loss": float(val_loss),
                        "val_perplexity": float(val_ppl),
                        "best_val_loss": float(min(best_val_loss, val_loss)),
                        "final": True,
                    })
                    wandb_log(
                        wandb_run,
                        {
                            "pretrain/val_loss": float(val_loss),
                            "pretrain/val_perplexity": float(val_ppl),
                            "pretrain/best_val_loss": float(min(best_val_loss, val_loss)),
                        },
                        step=global_step,
                    )
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        best_path = Path(args.out_dir) / "checkpoint_best.pt"
                        save_checkpoint(str(best_path), model, optimizer, global_step, epoch, args.tokenizer, asdict(config), best_val_loss)
                        print("Saved new best checkpoint:", best_path)
                final_path = Path(args.out_dir) / f"checkpoint_step_{global_step}.pt"
                save_checkpoint(str(final_path), model, optimizer, global_step, epoch, args.tokenizer, asdict(config), best_val_loss)
                print("\\nReached max_steps. Saved", final_path)
                wandb_finish(wandb_run)
                return

        # Save at the end of every epoch so progress survives Colab runtime resets.
        epoch_path = Path(args.out_dir) / f"checkpoint_step_{global_step}.pt"
        save_checkpoint(str(epoch_path), model, optimizer, global_step, epoch + 1, args.tokenizer, asdict(config), best_val_loss)
        print("\\nSaved end-of-epoch checkpoint:", epoch_path)

    if val_loader is not None and (args.eval_every <= 0 or global_step % args.eval_every != 0):
        val_loss = evaluate(model, val_loader, device, args.eval_batches)
        val_ppl = math.exp(min(val_loss, 20))
        append_jsonl(metrics_path, {
            "stage": "pretrain",
            "step": global_step,
            "epoch": args.epochs,
            "val_loss": float(val_loss),
            "val_perplexity": float(val_ppl),
            "best_val_loss": float(min(best_val_loss, val_loss)),
            "final": True,
        })
        wandb_log(
            wandb_run,
            {
                "pretrain/val_loss": float(val_loss),
                "pretrain/val_perplexity": float(val_ppl),
                "pretrain/best_val_loss": float(min(best_val_loss, val_loss)),
            },
            step=global_step,
        )
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = Path(args.out_dir) / "checkpoint_best.pt"
            save_checkpoint(str(best_path), model, optimizer, global_step, args.epochs, args.tokenizer, asdict(config), best_val_loss)
            print("Saved new best checkpoint:", best_path)

    final_path = Path(args.out_dir) / f"checkpoint_step_{global_step}.pt"
    save_checkpoint(str(final_path), model, optimizer, global_step, args.epochs, args.tokenizer, asdict(config), best_val_loss)
    print("Training complete. Saved", final_path)
    if best_val_loss < float("inf"):
        print(f"Best validation loss: {best_val_loss:.4f}")
    wandb_finish(wandb_run)


if __name__ == "__main__":
    main()


### 3.7 Generation Script - `src/generate.py`

Writes the base-model text generation utility.


In [ ]:
%%writefile src/generate.py
import argparse
from pathlib import Path

import torch
from tokenizers import Tokenizer

from model import GPT, GPTConfig
from utils import get_device, latest_checkpoint


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True)
    parser.add_argument("--checkpoint", default="checkpoints/pretrain/checkpoint_best.pt")
    parser.add_argument("--checkpoint_dir", default="checkpoints/pretrain")
    parser.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    parser.add_argument("--max_new_tokens", type=int, default=100)
    parser.add_argument("--temperature", type=float, default=0.8)
    parser.add_argument("--top_k", type=int, default=50)
    parser.add_argument("--greedy", action="store_true")
    args = parser.parse_args()

    checkpoint = args.checkpoint
    if not checkpoint or not Path(checkpoint).exists():
        checkpoint = latest_checkpoint(args.checkpoint_dir)
    if not checkpoint:
        raise FileNotFoundError(f"No checkpoint found in {args.checkpoint_dir}")

    device = get_device()
    state = torch.load(checkpoint, map_location=device)
    config = GPTConfig(**state["config"])

    tokenizer_path = args.tokenizer or state.get("tokenizer_path")
    tokenizer = Tokenizer.from_file(tokenizer_path)

    model = GPT(config).to(device)
    model.load_state_dict(state["model_state_dict"])
    model.eval()

    input_ids = tokenizer.encode(args.prompt).ids
    idx = torch.tensor([input_ids], dtype=torch.long, device=device)

    output = model.generate(
        idx,
        max_new_tokens=args.max_new_tokens,
        temperature=args.temperature,
        top_k=args.top_k,
        greedy=args.greedy,
    )
    text = tokenizer.decode(output[0].tolist(), skip_special_tokens=True)

    print("Checkpoint:", checkpoint)
    if "best_val_loss" in state:
        print("Best validation loss:", state["best_val_loss"])
    print("\nGenerated text:\n")
    print(text)


if __name__ == "__main__":
    main()

### 3.8 Pretraining README

Writes a short project README that mirrors the base pretraining workflow.


In [ ]:
%%writefile README_01_PRETRAIN.md
# TacticsGPT Phase 1: Pretraining

This project trains a small decoder-only GPT model on football tactical text using next-token prediction.

## Data

The provided Gemini article dump format is supported. Put the raw file at:

```text
data/raw_tactical_match_analysis.txt
```

Then build the cleaned corpus:

```bash
python src/build_corpus.py --raw data/raw_tactical_match_analysis.txt --out data/tactics_corpus.txt
```

This removes `ARTICLE #n` wrappers and separator bars while keeping match analysis, coach speeches, explanations, and tactical notes as training documents.

The cleaned text is written at:

```text
data/tactics_corpus.txt
```

Do not use JSON, CSV, labels, or question-answer pairs for Phase 1.

## Tokenizer

```bash
python src/tokenizer_train.py --corpus data/tactics_corpus.txt --out_dir checkpoints/tokenizer --vocab_size 8000
```

Outputs:

- `checkpoints/tokenizer/tokenizer.json`
- `checkpoints/tokenizer/vocab.json`
- `checkpoints/tokenizer/merges.txt`

## Pretraining

```bash
python src/train_pretrain.py --corpus data/tactics_corpus.txt --tokenizer checkpoints/tokenizer/tokenizer.json --epochs 30 --grad_accum_steps 2
```

The trainer automatically resumes from the latest `checkpoints/pretrain/checkpoint_step_*.pt` unless `--no_resume` is passed.

## Generation

```bash
python src/generate.py --prompt "How should a 4-4-2 defend centrally?"
```

This phase only trains tactical language modeling. It is not a RAG assistant and does not aim for factual perfection yet.

## 4. Train The Tokenizer

This step requires the cleaned corpus at `data/tactics_corpus.txt`.

Resume note: existing model checkpoints expect the same tokenizer they were trained with. If you are continuing old weights and `checkpoints/tokenizer/tokenizer.json` already exists, skip this cell unless you intentionally want to rebuild the tokenizer and retrain compatible weights.

If this cell fails, do not run training yet. Go back to **Add And Clean The Training Corpus**, upload the raw `.txt` file, and confirm it prints the cleaned document count.


In [ ]:
from pathlib import Path

corpus_file = Path("data/tactics_corpus.txt")
if not corpus_file.exists() or corpus_file.stat().st_size == 0:
    raise FileNotFoundError(
        "Missing data/tactics_corpus.txt.\n"
        "Run the 'Add The Training Corpus' cell first and upload tactical_match_analysis_5mb.txt.\n"
        "Do not use src/build_corpus.txt as the corpus path; src/build_corpus.py is only the cleaning script."
    )

print(f"Tokenizer input corpus: {corpus_file} ({corpus_file.stat().st_size:,} bytes)")
print("Tokenizer note: skip this when continuing old weights and tokenizer.json already exists.")
!python src/tokenizer_train.py --corpus data/tactics_corpus.txt --out_dir checkpoints/tokenizer --vocab_size 8000


## 5. Pretrain Or Continue The Base GPT

Run this only after tokenizer training succeeds and creates `checkpoints/tokenizer/tokenizer.json`.

Resume Disclaimer: this training script automatically loads the latest checkpoint in `checkpoints/pretrain`. It starts from zero only when no checkpoint exists or when `--no_resume` is added.

The run below is a full pretrain setup:

- `30` epochs
- no artificial `max_steps` cutoff
- checkpoint every `250` steps
- validation loss every `250` steps
- best checkpoint saved separately
- AMP enabled on CUDA
- gradient accumulation for a larger effective batch

If Colab runs out of GPU memory, reduce `--batch_size` to `8`, then `4`. Keep `--grad_accum_steps` at `2` or increase it to preserve the effective batch size.


In [ ]:
from pathlib import Path

corpus_file = Path("data/tactics_corpus.txt")
tokenizer_file = Path("checkpoints/tokenizer/tokenizer.json")

if not corpus_file.exists() or corpus_file.stat().st_size == 0:
    raise FileNotFoundError("Missing data/tactics_corpus.txt. Run the corpus upload/cleaning cell first.")
if not tokenizer_file.exists() or tokenizer_file.stat().st_size == 0:
    raise FileNotFoundError("Missing checkpoints/tokenizer/tokenizer.json. Run the tokenizer training cell first.")

print(f"Training corpus: {corpus_file} ({corpus_file.stat().st_size:,} bytes)")
print(f"Tokenizer: {tokenizer_file}")
print("Resume Disclaimer: pretraining loads the latest checkpoints/pretrain/checkpoint_step_*.pt unless --no_resume is added.")

!python src/train_pretrain.py \
  --corpus data/tactics_corpus.txt \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --epochs 30 \
  --batch_size 16 \
  --grad_accum_steps 2 \
  --context_length 256 \
  --stride 128 \
  --lr 3e-4 \
  --warmup_steps 500 \
  --save_every 250 \
  --eval_every 250 \
  --val_fraction 0.05 \
  --max_steps 0


## 6. Generate From The Base Model

Run this only after pretraining has saved at least one checkpoint in `checkpoints/pretrain`. This cell loads existing weights for inference only.


In [ ]:
from pathlib import Path

checkpoint_dir = Path("checkpoints/pretrain")
has_best = (checkpoint_dir / "checkpoint_best.pt").exists()
has_step = any(checkpoint_dir.glob("checkpoint_step_*.pt"))

if not has_best and not has_step:
    raise FileNotFoundError("No training checkpoint found. Run pretraining first and wait until it saves a checkpoint.")

!python src/generate.py \
  --prompt "How should a 4-4-2 defend centrally?" \
  --max_new_tokens 120 \
  --temperature 0.8 \
  --top_k 50

## 6.1 Optional Commands

Use these as small reference snippets for generation modes, intentional fresh starts, or larger model experiments.


In [ ]:
# Greedy decoding:
# !python src/generate.py --prompt "The midfield pivot must" --greedy --max_new_tokens 80

# Resume training happens automatically. To force a fresh run:
# !python src/train_pretrain.py --no_resume --max_steps 1000

# Larger model example, if your GPU has enough memory:
# !python src/train_pretrain.py --d_model 384 --n_heads 6 --n_layers 6 --batch_size 8

## 7. SFT LoRA Stage: Upload Dataset

Upload a JSONL file where each row has `instruction` and `response` fields. This creates `data/sft_dataset.jsonl`.


In [ ]:
from pathlib import Path
from google.colab import files

Path("data").mkdir(exist_ok=True)

print("Upload your SFT JSONL file with fields: instruction, response")
uploaded = files.upload()

name = next(iter(uploaded))
Path("data/sft_dataset.jsonl").write_bytes(uploaded[name])

print("Saved to data/sft_dataset.jsonl")

## 7.2 Write The LoRA SFT Trainer

This script loads the Phase 1 pretrained base checkpoint, applies LoRA layers, and automatically resumes the latest SFT adapter checkpoint in the selected SFT output folder.


In [ ]:
%%writefile src/train_sft_lora.py
import argparse, json, glob, os, re, math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

from model import GPT, GPTConfig
from wandb_utils import add_wandb_args, init_wandb, wandb_finish, wandb_log


def latest_step_checkpoint(folder, pattern):
    files = glob.glob(str(Path(folder) / pattern))
    if not files:
        return None
    def step(p):
        m = re.search(r"(\d+)\.pt$", os.path.basename(p))
        return int(m.group(1)) if m else -1
    return max(files, key=step)


def find_base_checkpoint():
    best = Path("checkpoints/pretrain/checkpoint_best.pt")
    if best.exists():
        return str(best)
    ckpt = latest_step_checkpoint("checkpoints/pretrain", "checkpoint_step_*.pt")
    if ckpt:
        return ckpt
    raise FileNotFoundError("No Phase 1 checkpoint found in checkpoints/pretrain")


class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        self.r = r
        self.scale = alpha / r
        self.dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)
        nn.init.zeros_(self.lora_B)

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + (self.dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scale


def apply_lora(model, r=8, alpha=16, dropout=0.05):
    for block in model.blocks:
        block.attn.qkv = LoRALinear(block.attn.qkv, r, alpha, dropout)
        block.attn.out = LoRALinear(block.attn.out, r, alpha, dropout)
        block.ffn.net[0] = LoRALinear(block.ffn.net[0], r, alpha, dropout)
        block.ffn.net[2] = LoRALinear(block.ffn.net[2], r, alpha, dropout)

    for name, p in model.named_parameters():
        p.requires_grad = "lora_" in name

    return model


def lora_state_dict(model):
    return {k: v.cpu() for k, v in model.state_dict().items() if "lora_" in k}


def sft_checkpoint_payload(model, optimizer, step, epoch, base_ckpt, config, best_eval_loss):
    return {
        "lora_state_dict": lora_state_dict(model),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "epoch": epoch,
        "base_checkpoint": base_ckpt,
        "config": config,
        "best_eval_loss": best_eval_loss,
    }


class SFTDataset(Dataset):
    def __init__(self, path, tokenizer_path, context_length=256):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.context_length = context_length
        self.pad_id = self.tokenizer.token_to_id("<pad>") or 0
        self.bos_id = self.tokenizer.token_to_id("<bos>")
        self.eos_id = self.tokenizer.token_to_id("<eos>")
        self.samples = []

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                instruction = obj["instruction"].strip()
                response = obj["response"].strip()

                prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
                prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False).ids
                response_ids = self.tokenizer.encode(response, add_special_tokens=False).ids

                ids = [self.bos_id] + prompt_ids + response_ids + [self.eos_id]
                labels = [-100] * (1 + len(prompt_ids)) + response_ids + [self.eos_id]

                ids = ids[:context_length]
                labels = labels[:context_length]

                # Causal LM shift:
                # x sees tokens up to t, y asks for token t+1.
                x_ids = ids[:-1]
                y_ids = labels[1:]

                if any(x != -100 for x in y_ids):
                    self.samples.append((x_ids, y_ids))

        if not self.samples:
            raise ValueError("No usable SFT samples found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

    def collate(self, batch):
        max_len = max(len(x[0]) for x in batch)
        xs, ys = [], []
        for ids, labels in batch:
            pad = max_len - len(ids)
            xs.append(ids + [self.pad_id] * pad)
            ys.append(labels + [-100] * pad)
        return torch.tensor(xs), torch.tensor(ys)


@torch.no_grad()
def evaluate(model, loader, device, max_batches):
    if loader is None:
        return None, None

    was_training = model.training
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=-100)
        losses.append(loss.item())

    if was_training:
        model.train()

    avg_loss = sum(losses) / max(1, len(losses))
    return avg_loss, math.exp(min(avg_loss, 20))


def append_jsonl(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sft_data", default="data/sft_dataset.jsonl")
    ap.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    ap.add_argument("--base_checkpoint", default="")
    ap.add_argument("--out_dir", default="checkpoints/sft")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--context_length", type=int, default=256)
    ap.add_argument("--save_every", type=int, default=100)
    ap.add_argument("--eval_every", type=int, default=100)
    ap.add_argument("--eval_batches", type=int, default=50)
    ap.add_argument("--val_fraction", type=float, default=0.05)
    ap.add_argument("--log_every", type=int, default=10)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--r", type=int, default=8)
    ap.add_argument("--alpha", type=int, default=16)
    ap.add_argument("--dropout", type=float, default=0.05)
    add_wandb_args(ap)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    Path(args.out_dir).mkdir(parents=True, exist_ok=True)
    metrics_path = Path(args.out_dir) / "metrics.jsonl"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    base_ckpt = args.base_checkpoint or find_base_checkpoint()
    print("Base checkpoint:", base_ckpt)

    state = torch.load(base_ckpt, map_location=device)
    config = GPTConfig(**state["config"])

    model = GPT(config).to(device)
    model.load_state_dict(state["model_state_dict"])
    model = apply_lora(model, args.r, args.alpha, args.dropout).to(device)

    dataset = SFTDataset(args.sft_data, args.tokenizer, args.context_length)
    val_size = max(1, int(len(dataset) * args.val_fraction)) if args.val_fraction > 0 and len(dataset) > 20 else 0
    train_size = len(dataset) - val_size
    if val_size:
        generator = torch.Generator().manual_seed(args.seed)
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)
    else:
        train_dataset = dataset
        val_dataset = None

    loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=dataset.collate)
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=dataset.collate)

    print(f"SFT samples: {len(dataset):,}")
    print(f"SFT train samples: {train_size:,}")
    print(f"SFT validation samples: {val_size:,}")

    wandb_run = init_wandb(
        args,
        "sft",
        {
            "stage": "sft",
            "args": vars(args),
            "base_checkpoint": base_ckpt,
            "model": state["config"],
            "samples": len(dataset),
            "train_samples": train_size,
            "val_samples": val_size,
        },
    )

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr)

    step = 0
    start_epoch = 0
    best_eval_loss = float("inf")
    resume = latest_step_checkpoint(args.out_dir, "sft_step_*.pt")
    if resume:
        print("Resuming SFT:", resume)
        ckpt = torch.load(resume, map_location=device)
        model.load_state_dict(ckpt["lora_state_dict"], strict=False)
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        step = ckpt["step"]
        start_epoch = ckpt["epoch"]
        best_eval_loss = ckpt.get("best_eval_loss", best_eval_loss)

    model.train()
    for epoch in range(start_epoch, args.epochs):
        pbar = tqdm(loader, desc=f"SFT epoch {epoch+1}/{args.epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)

            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=-100)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()

            step += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}", step=step)
            if args.log_every > 0 and step % args.log_every == 0:
                append_jsonl(metrics_path, {
                    "stage": "sft",
                    "step": step,
                    "epoch": epoch + 1,
                    "train_loss": float(loss.item()),
                    "lr": float(optimizer.param_groups[0]["lr"]),
                })
                wandb_log(
                    wandb_run,
                    {
                        "sft/train_loss": float(loss.item()),
                        "sft/lr": float(optimizer.param_groups[0]["lr"]),
                        "sft/epoch": epoch + 1,
                    },
                    step=step,
                )

            if val_loader is not None and args.eval_every > 0 and step % args.eval_every == 0:
                eval_loss, eval_ppl = evaluate(model, val_loader, device, args.eval_batches)
                print(f"\nSFT eval loss at step {step}: {eval_loss:.4f}")
                print(f"SFT eval perplexity at step {step}: {eval_ppl:.2f}")
                append_jsonl(metrics_path, {
                    "stage": "sft",
                    "step": step,
                    "epoch": epoch + 1,
                    "eval_loss": float(eval_loss),
                    "eval_perplexity": float(eval_ppl),
                    "best_eval_loss": float(min(best_eval_loss, eval_loss)),
                })
                wandb_log(
                    wandb_run,
                    {
                        "sft/eval_loss": float(eval_loss),
                        "sft/eval_perplexity": float(eval_ppl),
                        "sft/best_eval_loss": float(min(best_eval_loss, eval_loss)),
                    },
                    step=step,
                )
                if eval_loss < best_eval_loss:
                    best_eval_loss = eval_loss
                    best_path = Path(args.out_dir) / "sft_best.pt"
                    torch.save(
                        sft_checkpoint_payload(model, optimizer, step, epoch, base_ckpt, state["config"], best_eval_loss),
                        best_path,
                    )
                    print("Saved best SFT:", best_path)

            if step % args.save_every == 0:
                path = Path(args.out_dir) / f"sft_step_{step}.pt"
                torch.save(
                    sft_checkpoint_payload(model, optimizer, step, epoch, base_ckpt, state["config"], best_eval_loss),
                    path,
                )
                print("Saved", path)

    if val_loader is not None and (args.eval_every <= 0 or step % args.eval_every != 0):
        eval_loss, eval_ppl = evaluate(model, val_loader, device, args.eval_batches)
        print(f"\nFinal SFT eval loss: {eval_loss:.4f}")
        print(f"Final SFT perplexity: {eval_ppl:.2f}")
        append_jsonl(metrics_path, {
            "stage": "sft",
            "step": step,
            "epoch": args.epochs,
            "eval_loss": float(eval_loss),
            "eval_perplexity": float(eval_ppl),
            "best_eval_loss": float(min(best_eval_loss, eval_loss)),
            "final": True,
        })
        wandb_log(
            wandb_run,
            {
                "sft/eval_loss": float(eval_loss),
                "sft/eval_perplexity": float(eval_ppl),
                "sft/best_eval_loss": float(min(best_eval_loss, eval_loss)),
            },
            step=step,
        )
        if eval_loss < best_eval_loss:
            best_eval_loss = eval_loss
            best_path = Path(args.out_dir) / "sft_best.pt"
            torch.save(
                sft_checkpoint_payload(model, optimizer, step, args.epochs, base_ckpt, state["config"], best_eval_loss),
                best_path,
            )
            print("Saved best SFT:", best_path)

    final = Path(args.out_dir) / f"sft_step_{step}.pt"
    torch.save(
        sft_checkpoint_payload(model, optimizer, step, args.epochs, base_ckpt, state["config"], best_eval_loss),
        final,
    )
    print("Saved final SFT:", final)
    if best_eval_loss < float("inf"):
        print(f"Best SFT eval loss: {best_eval_loss:.4f}")
        print(f"Best SFT perplexity: {math.exp(min(best_eval_loss, 20)):.2f}")
    wandb_finish(wandb_run)


if __name__ == "__main__":
    main()


## 7.3 Run SFT: Smoke Test

Resume Disclaimer: this cell loads the pretrained base checkpoint first. If `checkpoints/sft/sft_step_*.pt` already exists, the trainer resumes the latest SFT adapter instead of starting SFT from zero.


In [ ]:
print("Resume Disclaimer: SFT loads the pretrained base and resumes latest checkpoints/sft/sft_step_*.pt if present.")
!python src/train_sft_lora.py \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --out_dir checkpoints/sft \
  --epochs 3 \
  --batch_size 8 \
  --lr 5e-5 \
  --context_length 256 \
  --save_every 50


## 7.4 Check Required SFT Files

Run this after mounting Drive if a later SFT cell cannot find the old corpus, tokenizer, base checkpoint, or trainer files.


In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/drive/MyDrive/TacticsGPT_Phase1_Full_Pretrain")
os.chdir(PROJECT_DIR)

required = [
    "data/sft_dataset.jsonl",
    "checkpoints/tokenizer/tokenizer.json",
    "checkpoints/pretrain/checkpoint_best.pt",
    "src/train_sft_lora.py",
    "src/model.py",
]

for p in required:
    path = Path(p)
    print(p, "OK" if path.exists() else "MISSING")

## 7.5 Continue SFT: Main Run

Resume Disclaimer: this uses the same `checkpoints/sft` folder as the smoke test, so it resumes the latest SFT adapter if one exists.


In [ ]:
print("Resume Disclaimer: SFT loads the pretrained base and resumes latest checkpoints/sft/sft_step_*.pt if present.")
!python src/train_sft_lora.py \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --out_dir checkpoints/sft \
  --epochs 20 \
  --batch_size 8 \
  --lr 1e-4 \
  --context_length 256 \
  --save_every 100


## 7.6 Write The SFT Evaluation Script

Creates a small evaluation script that loads the base model plus the latest SFT LoRA adapter.


In [ ]:
%%writefile src/evaluate_sft_lora.py
import argparse, glob, os, re, json, math
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tokenizers import Tokenizer

from model import GPT, GPTConfig
from train_sft_lora import apply_lora, SFTDataset


def latest_step_checkpoint(folder, pattern):
    files = glob.glob(str(Path(folder) / pattern))
    if not files:
        return None

    def step(p):
        m = re.search(r"(\d+)\.pt$", os.path.basename(p))
        return int(m.group(1)) if m else -1

    return max(files, key=step)


@torch.no_grad()
def eval_loss(model, dataset, batch_size, device, max_batches):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=dataset.collate)
    model.eval()

    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break

        x = x.to(device)
        y = y.to(device)

        logits, _ = model(x)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
            ignore_index=-100,
        )
        losses.append(loss.item())

    avg = sum(losses) / max(1, len(losses))
    return avg, math.exp(min(avg, 20))


@torch.no_grad()
def generate_answer(model, tokenizer, instruction, device, max_new_tokens=160, temperature=0.75, top_k=40):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    bos_id = tokenizer.token_to_id("<bos>")

    if bos_id is not None:
        ids = [bos_id] + ids

    x = torch.tensor([ids], dtype=torch.long, device=device)

    out = model.generate(
        x,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        greedy=False,
    )

    text = tokenizer.decode(out[0].tolist(), skip_special_tokens=True)

    if "### Response:" in text:
        text = text.split("### Response:", 1)[1].strip()

    return text


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sft_data", default="data/sft_dataset.jsonl")
    ap.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    ap.add_argument("--sft_dir", default="checkpoints/sft")
    ap.add_argument("--checkpoint", default="")
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--max_batches", type=int, default=50)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    sft_ckpt = args.checkpoint or latest_step_checkpoint(args.sft_dir, "sft_step_*.pt")
    if not sft_ckpt:
        raise FileNotFoundError("No SFT checkpoint found.")

    ckpt = torch.load(sft_ckpt, map_location=device)
    base_ckpt_path = ckpt["base_checkpoint"]

    print("SFT checkpoint:", sft_ckpt)
    print("Base checkpoint:", base_ckpt_path)

    base_state = torch.load(base_ckpt_path, map_location=device)
    config = GPTConfig(**base_state["config"])

    model = GPT(config).to(device)
    model.load_state_dict(base_state["model_state_dict"])

    model = apply_lora(model).to(device)
    model.load_state_dict(ckpt["lora_state_dict"], strict=False)

    tokenizer = Tokenizer.from_file(args.tokenizer)

    dataset = SFTDataset(args.sft_data, args.tokenizer, config.context_length)
    loss, ppl = eval_loss(model, dataset, args.batch_size, device, args.max_batches)

    print(f"\nSFT eval loss: {loss:.4f}")
    print(f"SFT perplexity: {ppl:.2f}")

    prompts = [
        "How should a 4-4-2 defend centrally against a 4-2-3-1?",
        "How can a team attack a compact 5-3-2 low block?",
        "What pressing cues should a 4-3-3 use against a back three?",
        "How should full-backs behave during defensive transitions?",
        "How should a team defend crosses in a low block?",
    ]

    for p in prompts:
        print("\n" + "=" * 80)
        print("INSTRUCTION:")
        print(p)
        print("\nMODEL RESPONSE:")
        print(generate_answer(model, tokenizer, p, device))


if __name__ == "__main__":
    main()

## 7.7 Evaluate The Latest SFT Adapter

This is load-only evaluation. It does not train or overwrite SFT weights.


In [ ]:
!python src/evaluate_sft_lora.py \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --sft_dir checkpoints/sft \
  --batch_size 8 \
  --max_batches 50

## 7.8 Continue SFT: Extended Run

Resume Disclaimer: this continues from the latest `checkpoints/sft/sft_step_*.pt` adapter when that file exists.


In [ ]:
print("Resume Disclaimer: SFT continues from latest checkpoints/sft/sft_step_*.pt if present.")
!python src/train_sft_lora.py \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --out_dir checkpoints/sft \
  --epochs 30 \
  --batch_size 8 \
  --lr 8e-5 \
  --context_length 256 \
  --save_every 100


## 8. Build RL Prompt File

Converts SFT instructions into unique RL prompts saved at `data/rl_prompts.jsonl`.


In [ ]:
import json
from pathlib import Path

sft_path = Path("data/sft_dataset.jsonl")
out = Path("data/rl_prompts.jsonl")

prompts = []
for line in sft_path.read_text(encoding="utf-8").splitlines():
    obj = json.loads(line)
    prompts.append(obj["instruction"].strip())

with out.open("w", encoding="utf-8") as f:
    for p in sorted(set(prompts)):
        f.write(json.dumps({"prompt": p}, ensure_ascii=False) + "\n")

print("Saved:", out)
print("Prompts:", len(set(prompts)))

## 9. Write The RL LoRA Trainer

This script loads the pretrained base checkpoint, loads the latest SFT adapter, and resumes the latest RL checkpoint in the selected RL output folder.


In [ ]:
%%writefile src/train_rl_lora.py
import argparse
import glob
import json
import math
import os
import random
import re
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from torch.utils.data import Dataset
from tqdm.auto import tqdm

from model import GPT, GPTConfig
from train_sft_lora import apply_lora, lora_state_dict
from wandb_utils import add_wandb_args, init_wandb, wandb_finish, wandb_log


ALLOWED_FORMATIONS = {
    "5-4-1",
    "5-2-1-2",
    "5-2-2-1",
    "5-2-3",
    "5-3-2",
    "3-1-4-2",
    "3-2-3-2",
    "3-2-4-1",
    "3-4-1-2",
    "3-4-2-1",
    "3-4-3",
    "3-5-1-1",
    "3-5-2",
    "4-1-2-1-2",
    "4-1-2-3",
    "4-1-3-2",
    "4-1-4-1",
    "4-2-1-3",
    "4-2-2-2",
    "4-2-3-1",
    "4-2-4",
    "4-3-1-2",
    "4-3-2-1",
    "4-3-3",
    "4-4-1-1",
    "4-4-2",
    "4-5-1",
}

DEFAULT_TACTICAL_TERMS = [
    "compact", "press", "pressing", "trigger", "block", "low block", "mid block",
    "high press", "transition", "counter", "half-space", "wide", "central",
    "overload", "underload", "pivot", "full-back", "wing-back", "centre-back",
    "center-back", "back line", "midfield", "cover shadow", "passing lane",
    "second ball", "rest defence", "rest defense", "numerical superiority",
    "depth", "width", "line", "back four", "back three", "third man",
]

TACTICAL_TERMS = DEFAULT_TACTICAL_TERMS

DEFAULT_ROLE_TERMS = [
    "full-back", "wing-back", "winger", "pivot", "centre-back", "center-back",
    "midfielder", "striker", "number 10", "holding midfielder", "back line",
    "front line", "wide player", "defensive midfielder",
]

ROLE_TERMS = DEFAULT_ROLE_TERMS

DEFAULT_ACTION_VERBS = [
    "screen", "delay", "press", "drop", "shift", "cover", "track", "force",
    "block", "protect", "hold", "mark", "trap", "recover", "switch", "stretch",
    "jump", "support", "occupy", "scan", "deny", "step", "pin", "rotate",
]

ACTION_VERBS = DEFAULT_ACTION_VERBS

PROMPT_GROUPS = {
    "press": ["press", "pressing", "trigger", "cover shadow", "jump", "force wide", "trap"],
    "transition": ["transition", "counter", "rest defence", "rest defense", "recover", "delay", "track runners"],
    "low_block": ["low block", "compact", "cross", "second ball", "box", "wide", "switch"],
    "build_up": ["build", "pivot", "centre-back", "center-back", "third man", "passing lane"],
    "cross": ["cross", "box", "near post", "far post", "second ball", "full-back", "winger"],
    "midfield": ["midfield", "pivot", "screen", "number 10", "central", "passing lane"],
}

STOPWORDS = {
    "about", "above", "after", "again", "against", "also", "because", "before",
    "being", "between", "could", "does", "doing", "down", "during", "each",
    "from", "have", "into", "more", "must", "need", "only", "other", "over",
    "same", "should", "than", "that", "their", "then", "there", "these",
    "they", "this", "those", "through", "under", "using", "very", "when",
    "where", "which", "while", "with", "would", "your", "team", "player",
    "players", "opponent", "opponents", "ball", "space", "spaces",
}

FORMATION_ANY_RE = re.compile(r"\b(?:[1-5]-){1,5}[1-5]\b")
BAD_REPETITION_RE = re.compile(r"\b(\w+)(?:\s+\1){3,}\b", re.IGNORECASE)
SYMBOL_SPAM_RE = re.compile(r"([.*_\-=])\1{6,}")
SPECIAL_TOKEN_RE = re.compile(r"(###|<bos>|<eos>|<pad>|instruction:|response:)", re.IGNORECASE)
WORD_RE = re.compile(r"[a-zA-Z][a-zA-Z\-']+")

GENERIC_BAD_PHRASES = [
    "you must be fluid",
    "specific zones of the pitch",
    "act as the pivot point of the ball carrier",
    "the ball is central",
    "to prevent the opponent from behind",
    "maintain the anchor",
    "players should work together",
    "keep good shape",
]


def normalize_prompt(text):
    return " ".join(text.lower().strip().split())


def words(text):
    return WORD_RE.findall(text.lower())


def content_words(text):
    return [w for w in words(text) if len(w) >= 4 and w not in STOPWORDS]


def sentence_count(text):
    return len([s for s in re.split(r"[.!?]+", text) if len(s.strip()) > 12])


def unique_word_ratio(text):
    w = words(text)
    return len(set(w)) / max(1, len(w))


def tactical_terms():
    return globals().get("TACTICAL_TERMS", DEFAULT_TACTICAL_TERMS)


def role_terms():
    return globals().get("ROLE_TERMS", DEFAULT_ROLE_TERMS)


def action_verbs():
    return globals().get("ACTION_VERBS", DEFAULT_ACTION_VERBS)


def extract_key_terms(text, max_terms=32):
    lower = text.lower()
    terms = set()

    for term in tactical_terms() + role_terms() + action_verbs():
        if term in lower:
            terms.add(term)

    counts = Counter(content_words(text))
    for word, _ in counts.most_common(max_terms):
        terms.add(word)

    cw = content_words(text)
    phrase_counts = Counter()
    for n in (2, 3):
        for i in range(0, max(0, len(cw) - n + 1)):
            phrase = " ".join(cw[i:i + n])
            if len(phrase) >= 9:
                phrase_counts[phrase] += 1

    for phrase, _ in phrase_counts.most_common(10):
        terms.add(phrase)

    return terms


def parse_formations(text):
    formations = []
    for match in FORMATION_ANY_RE.finditer(text):
        raw = match.group(0)
        parts = [int(x) for x in raw.split("-")]
        formations.append((raw, parts, sum(parts)))
    return formations


def formation_score(response):
    formations = parse_formations(response)
    if not formations:
        return 0.0, ["no_formation"]

    invalid = []
    valid = []
    for raw, parts, total in formations:
        if raw in ALLOWED_FORMATIONS and total == 10:
            valid.append(raw)
        else:
            invalid.append(raw)

    if invalid:
        return -1.8, [f"invalid_formation:{','.join(invalid[:3])}"]
    return 0.35, [f"valid_formation:{valid[0]}"]


def prompt_focus_groups(prompt):
    lower = prompt.lower()
    return [group for group, keys in PROMPT_GROUPS.items() if any(k in lower for k in keys)]


def prompt_relevance_score(prompt, response):
    lower = response.lower()
    prompt_terms = set(content_words(prompt))
    if not prompt_terms:
        prompt_overlap = 0.0
    else:
        prompt_overlap = len([w for w in prompt_terms if w in lower]) / max(2, min(6, len(prompt_terms)))
        prompt_overlap = min(1.0, prompt_overlap)

    groups = prompt_focus_groups(prompt)
    if not groups:
        return prompt_overlap

    group_scores = []
    for group in groups:
        keys = PROMPT_GROUPS[group]
        hits = sum(1 for key in keys if key in lower)
        group_scores.append(min(1.0, hits / 3))

    group_score = sum(group_scores) / max(1, len(group_scores))
    return 0.55 * group_score + 0.45 * prompt_overlap


def load_reference_index(path):
    ref_path = Path(path)
    if not ref_path.exists():
        print(f"Reference note: {ref_path} not found, SFT answer matching disabled.")
        return {}

    index = {}
    for line in ref_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        prompt = obj.get("instruction") or obj.get("prompt") or obj.get("question")
        answer = obj.get("response") or obj.get("answer") or obj.get("completion")
        if not prompt or not answer:
            continue

        key = normalize_prompt(prompt)
        index.setdefault(key, []).append({
            "answer": answer.strip(),
            "terms": extract_key_terms(answer),
        })

    print(f"Loaded reference answers: {len(index)} prompts from {ref_path}")
    return index


def reference_alignment_score(prompt, response, reference_index):
    refs = reference_index.get(normalize_prompt(prompt), [])
    if not refs:
        return 0.0, ["no_reference"]

    response_terms = extract_key_terms(response)
    if not response_terms:
        return 0.0, ["empty_response_terms"]

    best = 0.0
    best_shared = []
    for ref in refs:
        ref_terms = set(ref["terms"])
        if not ref_terms:
            continue
        shared = ref_terms & response_terms
        coverage = len(shared) / max(1, len(ref_terms))
        precision = len(shared) / max(1, len(response_terms))
        score = 0.75 * coverage + 0.25 * precision
        if score > best:
            best = score
            best_shared = sorted(shared)[:8]

    return min(1.0, best), [f"reference_overlap:{','.join(best_shared)}"] if best_shared else ["no_reference_overlap"]


def formatting_score(response):
    text = response.strip()
    lower = text.lower()
    score = 0.0
    reasons = []

    if not text:
        return -1.5, ["empty_format"]

    if SPECIAL_TOKEN_RE.search(text):
        score -= 1.0
        reasons.append("leaked_prompt_tokens")

    if text[-1] in ".!?":
        score += 0.15
        reasons.append("clean_ending")
    else:
        score -= 0.35
        reasons.append("unfinished_sentence")

    sentences = sentence_count(text)
    if 2 <= sentences <= 6:
        score += 0.25
        reasons.append("clear_sentence_count")
    elif sentences == 0:
        score -= 0.5
        reasons.append("no_clear_sentences")

    if re.search(r"(^|\n)\s*(-|\d+\.)\s+\w+", text):
        score += 0.15
        reasons.append("structured_list")

    if any(connector in lower for connector in ["because", "so that", "which forces", "when", "if", "therefore"]):
        score += 0.2
        reasons.append("clear_cause_effect")

    if re.search(r"([!?.,])\1{3,}", text):
        score -= 0.4
        reasons.append("punctuation_spam")

    return score, reasons


def tactical_logic_score(response):
    lower = response.lower()
    role_hit = any(role in lower for role in role_terms())
    action_hit = any(action in lower for action in action_verbs())
    outcome_hit = any(x in lower for x in [
        "so that", "because", "which forces", "to prevent", "to deny",
        "to protect", "when", "if", "therefore", "as a result",
    ])

    if role_hit and action_hit and outcome_hit:
        return 0.9, ["role_action_outcome"]
    if action_hit and outcome_hit:
        return 0.35, ["action_outcome"]
    return -0.55, ["weak_tactical_logic"]


def repetition_penalty(response):
    lower = response.lower()
    penalty = 0.0
    reasons = []

    if BAD_REPETITION_RE.search(lower):
        penalty -= 1.2
        reasons.append("word_loop")

    if SYMBOL_SPAM_RE.search(response):
        penalty -= 1.2
        reasons.append("symbol_spam")

    w = words(response)
    if len(w) > 20:
        trigrams = list(zip(w, w[1:], w[2:]))
        if trigrams:
            repeated_ratio = 1.0 - len(set(trigrams)) / len(trigrams)
            if repeated_ratio > 0.18:
                penalty -= min(1.2, repeated_ratio * 3.0)
                reasons.append("repeated_trigrams")

        lower_counts = Counter(w)
        for word, count in lower_counts.items():
            if len(word) > 4 and count >= 7:
                penalty -= 0.2 * (count - 6)
                reasons.append(f"repeated_word:{word}")
                break

    return penalty, reasons


def generic_answer_penalty(response):
    lower = response.lower()
    penalty = 0.0
    reasons = []

    for phrase in GENERIC_BAD_PHRASES:
        if phrase in lower:
            penalty -= 0.6
            reasons.append(f"generic_phrase:{phrase[:20]}")

    if unique_word_ratio(response) < 0.42 and len(words(response)) > 45:
        penalty -= 0.7
        reasons.append("low_diversity")

    return penalty, reasons


def reward_response(prompt, response, reference_index=None):
    reference_index = reference_index or {}
    text = response.strip()
    w = words(text)
    n_words = len(w)
    reward = 0.0
    reasons = []

    if not text:
        return -4.0, ["empty_response"]

    rep_penalty, rep_reasons = repetition_penalty(text)
    if rep_penalty <= -2.0:
        return -4.0, ["collapse_repetition"] + rep_reasons

    if n_words < 35:
        reward -= 1.2
        reasons.append("too_short")
    elif 55 <= n_words <= 200:
        reward += 0.45
        reasons.append("good_length")
    elif 35 <= n_words < 55 or 200 < n_words <= 280:
        reward += 0.05
        reasons.append("acceptable_length")
    else:
        reward -= 0.6
        reasons.append("bad_length")

    rel = prompt_relevance_score(prompt, text)
    if rel >= 0.62:
        reward += 1.1
        reasons.append("prompt_relevant")
    elif rel >= 0.35:
        reward += 0.35
        reasons.append("partly_relevant")
    else:
        reward -= 1.1
        reasons.append("off_prompt")

    ref_score, ref_reasons = reference_alignment_score(prompt, text, reference_index)
    if reference_index:
        if ref_score >= 0.42:
            reward += 1.25
            reasons.append("strong_reference_alignment")
        elif ref_score >= 0.22:
            reward += 0.5
            reasons.append("some_reference_alignment")
        elif ref_score > 0.0:
            reward += 0.1
            reasons.append("thin_reference_alignment")
        else:
            reward -= 0.7
            reasons.append("missing_reference_concepts")
        reasons.extend(ref_reasons)

    f_score, f_reasons = formation_score(text)
    reward += f_score
    reasons.extend(f_reasons)

    fmt_score, fmt_reasons = formatting_score(text)
    reward += fmt_score
    reasons.extend(fmt_reasons)

    logic_score, logic_reasons = tactical_logic_score(text)
    reward += logic_score
    reasons.extend(logic_reasons)

    term_hits = [term for term in tactical_terms() if term in text.lower()]
    if 3 <= len(term_hits) <= 10:
        reward += 0.35
        reasons.append("balanced_tactical_terms")
    elif len(term_hits) > 13:
        reward -= 0.65
        reasons.append("keyword_stuffing")
    elif len(term_hits) < 2:
        reward -= 0.25
        reasons.append("thin_tactical_terms")

    reward += rep_penalty
    reasons.extend(rep_reasons)

    generic_penalty, generic_reasons = generic_answer_penalty(text)
    reward += generic_penalty
    reasons.extend(generic_reasons)

    return max(-4.0, min(5.0, reward)), reasons


class RLPromptDataset(Dataset):
    def __init__(self, rl_path, fallback_sft_path=None):
        self.prompts = []
        path = Path(rl_path)

        if path.exists():
            for line in path.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                obj = json.loads(line)
                prompt = obj.get("prompt") or obj.get("instruction") or obj.get("question")
                if prompt:
                    self.prompts.append(prompt.strip())

        if not self.prompts and fallback_sft_path and Path(fallback_sft_path).exists():
            for line in Path(fallback_sft_path).read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                obj = json.loads(line)
                prompt = obj.get("instruction") or obj.get("prompt") or obj.get("question")
                if prompt:
                    self.prompts.append(prompt.strip())

        self.prompts = sorted(set(self.prompts))
        if not self.prompts:
            raise ValueError(f"No RL prompts found in {rl_path} or fallback SFT data.")

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        return self.prompts[idx]


def latest_step_checkpoint(folder, pattern):
    files = glob.glob(str(Path(folder) / pattern))
    if not files:
        return None

    def step(path):
        match = re.search(r"(\d+)\.pt$", os.path.basename(path))
        return int(match.group(1)) if match else -1

    return max(files, key=step)


def find_base_checkpoint():
    best = Path("checkpoints/pretrain/checkpoint_best.pt")
    if best.exists():
        return str(best)
    ckpt = latest_step_checkpoint("checkpoints/pretrain", "checkpoint_step_*.pt")
    if ckpt:
        return ckpt
    raise FileNotFoundError("No Phase 1 checkpoint found.")


def find_sft_checkpoint():
    ckpt = latest_step_checkpoint("checkpoints/sft", "sft_step_*.pt")
    if ckpt:
        return ckpt
    raise FileNotFoundError("No SFT checkpoint found. Train SFT before RL.")


def sample_with_logprobs(model, tokenizer, instruction, device, max_new_tokens=120, temperature=0.8, top_k=40):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")

    if bos_id is not None:
        ids = [bos_id] + ids

    x = torch.tensor([ids], dtype=torch.long, device=device)
    log_probs = []
    generated = []

    model.train()
    for _ in range(max_new_tokens):
        x_cond = x[:, -model.config.context_length:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-6)

        if top_k and top_k > 0:
            values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits = logits.masked_fill(logits < values[:, [-1]], float("-inf"))

        probs = F.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs=probs)
        next_id = dist.sample()
        log_probs.append(dist.log_prob(next_id))

        token_id = int(next_id.item())
        generated.append(token_id)
        x = torch.cat([x, next_id.view(1, 1)], dim=1)

        if eos_id is not None and token_id == eos_id:
            break

    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    if not log_probs:
        return response, torch.tensor(0.0, device=device), 1

    return response, torch.stack(log_probs).sum(), max(1, len(generated))


@torch.no_grad()
def generate_response(model, tokenizer, instruction, device, max_new_tokens=140, temperature=0.7, top_k=40):
    was_training = model.training
    model.eval()

    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")

    if bos_id is not None:
        ids = [bos_id] + ids

    x = torch.tensor([ids], dtype=torch.long, device=device)
    generated = []

    for _ in range(max_new_tokens):
        x_cond = x[:, -model.config.context_length:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-6)

        if top_k and top_k > 0:
            values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits = logits.masked_fill(logits < values[:, [-1]], float("-inf"))

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        token_id = int(next_id.item())
        generated.append(token_id)
        x = torch.cat([x, next_id.view(1, 1)], dim=1)

        if eos_id is not None and token_id == eos_id:
            break

    if was_training:
        model.train()

    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def evaluate_policy(model, tokenizer, prompts, reference_index, device, args):
    sample_size = min(args.eval_samples, len(prompts))
    eval_prompts = random.sample(prompts, sample_size) if sample_size < len(prompts) else list(prompts)

    rewards = []
    reason_counts = Counter()
    examples = []

    for prompt in eval_prompts:
        response = generate_response(
            model,
            tokenizer,
            prompt,
            device,
            max_new_tokens=args.max_new_tokens,
            temperature=args.eval_temperature,
            top_k=args.eval_top_k,
        )
        reward, reasons = reward_response(prompt, response, reference_index)
        rewards.append(float(reward))
        reason_counts.update(reasons)
        if len(examples) < 3:
            examples.append({
                "prompt": prompt,
                "response": response[:600],
                "reward": float(reward),
                "reasons": reasons[:12],
            })

    avg_reward = sum(rewards) / max(1, len(rewards))
    return {
        "avg_reward": avg_reward,
        "min_reward": min(rewards) if rewards else 0.0,
        "max_reward": max(rewards) if rewards else 0.0,
        "samples": len(rewards),
        "top_reasons": reason_counts.most_common(12),
        "examples": examples,
    }


def checkpoint_payload(model, optimizer, step, base_path, sft_path, base_config, best_eval_reward,
                       reward_history, eval_history, top_k, args):
    return {
        "lora_state_dict": lora_state_dict(model),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "best_eval_reward": best_eval_reward,
        "reward_history": reward_history[-1000:],
        "eval_history": eval_history[-200:],
        "top_k": top_k,
        "base_checkpoint": base_path,
        "sft_checkpoint": sft_path,
        "config": base_config,
        "args": vars(args),
    }


def save_training_checkpoint(path, model, optimizer, step, base_path, sft_path, base_config,
                             best_eval_reward, reward_history, eval_history, top_k, args):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        checkpoint_payload(
            model, optimizer, step, base_path, sft_path, base_config,
            best_eval_reward, reward_history, eval_history, top_k, args,
        ),
        path,
    )
    return str(path)


def update_top_k(top_k, score, path, k):
    if k <= 0:
        return []

    path = str(path)
    top_k = [item for item in top_k if item.get("path") != path]
    top_k.append({"score": float(score), "path": path})
    top_k = sorted(top_k, key=lambda item: item["score"], reverse=True)

    removed = top_k[k:]
    kept = top_k[:k]

    for item in removed:
        old_path = Path(item["path"])
        if old_path.exists() and old_path.name != "checkpoint_best.pt":
            try:
                old_path.unlink()
            except OSError:
                pass

    return kept


def append_jsonl(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def reward_improvement_pct(current_reward, baseline_reward):
    if baseline_reward is None or not math.isfinite(baseline_reward):
        return None
    if abs(baseline_reward) < 1e-8:
        return None
    return 100.0 * (current_reward - baseline_reward) / abs(baseline_reward)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rl_data", default="data/rl_prompts.jsonl")
    ap.add_argument("--sft_data", default="data/sft_dataset.jsonl")
    ap.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    ap.add_argument("--base_checkpoint", default="")
    ap.add_argument("--sft_checkpoint", default="")
    ap.add_argument("--resume_checkpoint", default="")
    ap.add_argument("--out_dir", default="checkpoints/rl")
    ap.add_argument("--steps", type=int, default=500)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--batch_size", type=int, default=1)
    ap.add_argument("--group_size", type=int, default=2)
    ap.add_argument("--save_every", type=int, default=50)
    ap.add_argument("--eval_every", type=int, default=50)
    ap.add_argument("--eval_samples", type=int, default=16)
    ap.add_argument("--top_k_best", type=int, default=1)
    ap.add_argument("--min_delta", type=float, default=1e-4)
    ap.add_argument("--log_every", type=int, default=10)
    ap.add_argument("--max_new_tokens", type=int, default=120)
    ap.add_argument("--temperature", type=float, default=0.8)
    ap.add_argument("--top_k", type=int, default=40)
    ap.add_argument("--eval_temperature", type=float, default=0.65)
    ap.add_argument("--eval_top_k", type=int, default=40)
    ap.add_argument("--seed", type=int, default=42)
    add_wandb_args(ap)
    args = ap.parse_args()

    random.seed(args.seed)
    torch.manual_seed(args.seed)

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.jsonl"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    base_path = args.base_checkpoint or find_base_checkpoint()
    sft_path = args.sft_checkpoint or find_sft_checkpoint()

    print("Device:", device)
    print("Base:", base_path)
    print("SFT:", sft_path)
    print("Reward mode: GRPO-style group-relative advantage, SFT-reference keyword matching, strict optional formations.")

    wandb_run = init_wandb(
        args,
        "rl_grpo",
        {
            "stage": "rl_grpo",
            "args": vars(args),
            "base_checkpoint": base_path,
            "sft_checkpoint": sft_path,
        },
    )

    base_state = torch.load(base_path, map_location=device)
    sft_state = torch.load(sft_path, map_location=device)

    config = GPTConfig(**base_state["config"])
    model = GPT(config).to(device)
    model.load_state_dict(base_state["model_state_dict"])

    model = apply_lora(model).to(device)
    model.load_state_dict(sft_state["lora_state_dict"], strict=False)

    for _, param in model.named_parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if "lora_" in name:
            param.requires_grad = True

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr)
    tokenizer = Tokenizer.from_file(args.tokenizer)
    dataset = RLPromptDataset(args.rl_data, fallback_sft_path=args.sft_data)
    reference_index = load_reference_index(args.sft_data)

    step = 0
    baseline = 0.0
    best_eval_reward = -float("inf")
    reward_history = []
    eval_history = []
    top_k = []
    baseline_eval_reward = None

    resume = args.resume_checkpoint or latest_step_checkpoint(args.out_dir, "rl_step_*.pt")
    if resume:
        print("Resuming RL:", resume)
        ckpt = torch.load(resume, map_location=device)
        model.load_state_dict(ckpt["lora_state_dict"], strict=False)
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        step = ckpt.get("step", 0)
        baseline = ckpt.get("baseline", 0.0)
        best_eval_reward = ckpt.get("best_eval_reward", ckpt.get("best_avg_reward", best_eval_reward))
        reward_history = ckpt.get("reward_history", [])
        eval_history = ckpt.get("eval_history", [])
        top_k = ckpt.get("top_k", [])
        if eval_history:
            baseline_eval_reward = float(eval_history[0].get("avg_reward", eval_history[0].get("eval_reward")))
        print("Resume state:", {"step": step, "best_eval_reward": best_eval_reward, "top_k": top_k})

    pbar = tqdm(total=args.steps, initial=step, desc="RL GRPO-style")

    while step < args.steps:
        batch_losses = []
        batch_rewards = []
        batch_records = []

        for _ in range(args.batch_size):
            prompt = random.choice(dataset.prompts)
            group = []

            for _ in range(max(1, args.group_size)):
                response, log_prob_sum, token_count = sample_with_logprobs(
                    model,
                    tokenizer,
                    prompt,
                    device,
                    max_new_tokens=args.max_new_tokens,
                    temperature=args.temperature,
                    top_k=args.top_k,
                )
                reward, reasons = reward_response(prompt, response, reference_index)
                group.append({
                    "prompt": prompt,
                    "response": response,
                    "log_prob_sum": log_prob_sum,
                    "token_count": token_count,
                    "reward": float(reward),
                    "reasons": reasons,
                })

            rewards_tensor = torch.tensor([item["reward"] for item in group], dtype=torch.float32, device=device)
            if len(group) > 1:
                mean = rewards_tensor.mean()
                std = rewards_tensor.std(unbiased=False)
                advantages = (rewards_tensor - mean) / (std + 1e-6)
            else:
                reward_value = float(rewards_tensor.item())
                baseline = 0.95 * baseline + 0.05 * reward_value
                advantages = torch.tensor([reward_value - baseline], dtype=torch.float32, device=device)

            for item, advantage in zip(group, advantages):
                normalized_logprob = item["log_prob_sum"] / max(1, item["token_count"])
                batch_losses.append(-normalized_logprob * advantage.detach())
                batch_rewards.append(item["reward"])
                batch_records.append(item)

        loss = torch.stack(batch_losses).mean()
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()

        step += 1
        current_reward = float(batch_rewards[-1])
        avg_train_reward = sum(batch_rewards) / max(1, len(batch_rewards))
        reward_history.append(avg_train_reward)
        reward_history = reward_history[-1000:]
        recent_avg = sum(reward_history[-25:]) / max(1, len(reward_history[-25:]))

        checkpoint_path = ""
        best_checkpoint_path = ""
        eval_result = None

        if step % args.save_every == 0:
            checkpoint_path = save_training_checkpoint(
                out_dir / f"rl_step_{step}.pt",
                model,
                optimizer,
                step,
                base_path,
                sft_path,
                base_state["config"],
                best_eval_reward,
                reward_history,
                eval_history,
                top_k,
                args,
            )
            print(f"\nSaved recovery checkpoint: {checkpoint_path}")

        if args.eval_every > 0 and step % args.eval_every == 0:
            eval_result = evaluate_policy(model, tokenizer, dataset.prompts, reference_index, device, args)
            eval_reward = float(eval_result["avg_reward"])
            if baseline_eval_reward is None:
                baseline_eval_reward = eval_reward
            improvement = reward_improvement_pct(eval_reward, baseline_eval_reward)
            eval_history.append({"step": step, "reward_improvement_pct": improvement, **eval_result})

            if eval_reward > best_eval_reward + args.min_delta:
                best_eval_reward = eval_reward
                best_checkpoint_path = save_training_checkpoint(
                    out_dir / "best_model" / "checkpoint_best.pt",
                    model,
                    optimizer,
                    step,
                    base_path,
                    sft_path,
                    base_state["config"],
                    best_eval_reward,
                    reward_history,
                    eval_history,
                    top_k,
                    args,
                )

                if args.top_k_best > 1:
                    top_k_path = out_dir / "best_model" / "top_k" / f"best_step_{step}_reward_{eval_reward:.4f}.pt"
                    saved_top_k_path = save_training_checkpoint(
                        top_k_path,
                        model,
                        optimizer,
                        step,
                        base_path,
                        sft_path,
                        base_state["config"],
                        best_eval_reward,
                        reward_history,
                        eval_history,
                        top_k,
                        args,
                    )
                    top_k = update_top_k(top_k, eval_reward, saved_top_k_path, args.top_k_best)

                print(f"\nSaved new best model: {best_checkpoint_path} eval_reward={eval_reward:.4f}")

        if step % args.log_every == 0 or checkpoint_path or best_checkpoint_path:
            record = {
                "step": step,
                "current_reward": current_reward,
                "avg_train_reward": avg_train_reward,
                "recent_avg_reward_25": recent_avg,
                "best_eval_reward": best_eval_reward,
                "loss": float(loss.item()),
                "checkpoint_path": checkpoint_path,
                "best_checkpoint_path": best_checkpoint_path,
                "sample_prompt": batch_records[-1]["prompt"] if batch_records else "",
                "sample_response": batch_records[-1]["response"][:500] if batch_records else "",
                "sample_reasons": batch_records[-1]["reasons"][:12] if batch_records else [],
            }
            if eval_result:
                record["eval_reward"] = eval_result["avg_reward"]
                record["eval_top_reasons"] = eval_result["top_reasons"]
                record["reward_improvement_pct"] = reward_improvement_pct(
                    float(eval_result["avg_reward"]),
                    baseline_eval_reward,
                )
            append_jsonl(metrics_path, record)
            wandb_record = {
                "rl/loss": float(loss.item()),
                "rl/lr": float(optimizer.param_groups[0]["lr"]),
                "rl/current_reward": current_reward,
                "rl/avg_train_reward": avg_train_reward,
                "rl/recent_avg_reward_25": recent_avg,
            }
            if math.isfinite(best_eval_reward):
                wandb_record["rl/best_eval_reward"] = best_eval_reward
            if eval_result:
                wandb_record.update({
                    "rl/eval_reward": float(eval_result["avg_reward"]),
                    "rl/eval_min_reward": float(eval_result["min_reward"]),
                    "rl/eval_max_reward": float(eval_result["max_reward"]),
                    "rl/eval_samples": int(eval_result["samples"]),
                })
                improvement = reward_improvement_pct(float(eval_result["avg_reward"]), baseline_eval_reward)
                if improvement is not None:
                    wandb_record["rl/reward_improvement_pct"] = float(improvement)
            wandb_log(wandb_run, wandb_record, step=step)

            print(
                "\nLOG",
                json.dumps({
                    "step": step,
                    "current_reward": round(current_reward, 3),
                    "avg_train_reward": round(avg_train_reward, 3),
                    "recent_avg25": round(recent_avg, 3),
                    "best_eval_reward": round(best_eval_reward, 3) if math.isfinite(best_eval_reward) else None,
                    "checkpoint_path": checkpoint_path,
                    "best_checkpoint_path": best_checkpoint_path,
                }),
            )

        pbar.update(1)
        pbar.set_postfix(
            reward=f"{current_reward:.2f}",
            avg=f"{avg_train_reward:.2f}",
            avg25=f"{recent_avg:.2f}",
            best=f"{best_eval_reward:.2f}" if math.isfinite(best_eval_reward) else "none",
            loss=f"{loss.item():.4f}",
        )

    final_path = save_training_checkpoint(
        out_dir / f"rl_step_{step}.pt",
        model,
        optimizer,
        step,
        base_path,
        sft_path,
        base_state["config"],
        best_eval_reward,
        reward_history,
        eval_history,
        top_k,
        args,
    )
    print("Saved final RL:", final_path)
    print("Metrics:", metrics_path)
    if math.isfinite(best_eval_reward):
        print("Best eval reward:", best_eval_reward)
        print("Best model:", out_dir / "best_model" / "checkpoint_best.pt")
        if baseline_eval_reward is not None:
            improvement = reward_improvement_pct(best_eval_reward, baseline_eval_reward)
            if improvement is not None:
                print(f"Best reward improvement: {improvement:.2f}%")
    wandb_finish(wandb_run)


if __name__ == "__main__":
    main()


## 10. Train Or Continue RL

This is the only RL training cell you need.

Resume rule: keep `RL_OUT_DIR` the same when continuing. The trainer resumes from the latest `rl_step_*.pt` inside that folder.

If your first run used `TOTAL_STEPS = 200`, continue by increasing `TOTAL_STEPS` to a larger number, for example `1000`, while keeping `RL_OUT_DIR = "checkpoints/rl"`.


In [ ]:
# Single RL training/continue cell
# Keep RL_OUT_DIR the same across runs. Increase TOTAL_STEPS to continue farther.

RL_OUT_DIR = "checkpoints/rl"
TOTAL_STEPS = 1000

print(f"Resume Disclaimer: RL resumes latest {RL_OUT_DIR}/rl_step_*.pt if present.")
print(f"Training target: continue until step {TOTAL_STEPS}. If already at/above that step, increase TOTAL_STEPS.")

!python src/train_rl_lora.py \
  --rl_data data/rl_prompts.jsonl \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --out_dir {RL_OUT_DIR} \
  --steps {TOTAL_STEPS} \
  --lr 5e-5 \
  --save_every 25 \
  --eval_every 25 \
  --eval_samples 8 \
  --group_size 2 \
  --top_k_best 3 \
  --max_new_tokens 120 \
  --temperature 0.75 \
  --top_k 40


## 11. Manual Test Fixed Prompts

Load-only test cell. It uses the same `checkpoints/rl` folder as the RL training cell and generates answers for a small prompt list.


In [ ]:
# Manual fixed-prompt test cell
# Load-only: this does not train or overwrite checkpoints.

from pathlib import Path
import glob
import os
import re
import sys
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/TacticsGPT_Phase1_Full_Pretrain")
if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)

SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tokenizers import Tokenizer
from model import GPT, GPTConfig
from train_sft_lora import apply_lora

ADAPTER_DIR = "checkpoints/rl"   # same folder as the RL training cell
# ADAPTER_DIR = "checkpoints/sft" # use this to test SFT only

TOKENIZER_PATH = "checkpoints/tokenizer/tokenizer.json"
BASE_CHECKPOINT = "checkpoints/pretrain/checkpoint_best.pt"

TEST_PROMPTS = [
    "How should a 4-4-2 defend central spaces against a 4-2-3-1?",
    "What pressing triggers should a team use against short build-up?",
    "How should a low block defend crosses from wide areas?",
]


def latest_ckpt(folder, patterns):
    if isinstance(patterns, str):
        patterns = [patterns]

    files = []
    for pattern in patterns:
        files.extend(glob.glob(str(Path(folder) / pattern)))

    if not files:
        return None

    def score(path):
        path = Path(path)
        if path.name == "checkpoint_best.pt" and path.parent.name == "best_model":
            return 10**18
        if path.name == "checkpoint_best_reward.pt":
            return 10**18 - 1
        match = re.search(r"(\d+)\.pt$", path.name)
        return int(match.group(1)) if match else -1

    return max(files, key=score)


def find_base_checkpoint():
    if Path(BASE_CHECKPOINT).exists():
        return BASE_CHECKPOINT
    ckpt = latest_ckpt("checkpoints/pretrain", "checkpoint_step_*.pt")
    if ckpt:
        return ckpt
    raise FileNotFoundError("No base checkpoint found.")


def find_adapter_checkpoint(adapter_dir):
    adapter_dir = Path(adapter_dir)
    patterns = [
        "best_model/checkpoint_best.pt",
        "checkpoint_best_reward.pt",
        "rl_step_*.pt",
        "sft_step_*.pt",
    ]
    ckpt = latest_ckpt(adapter_dir, patterns)
    if ckpt:
        return ckpt
    raise FileNotFoundError(f"No adapter checkpoint found in {adapter_dir}")


@torch.no_grad()
def answer(model, tokenizer, instruction, device, max_new_tokens=180, temperature=0.7, top_k=40):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    ids = tokenizer.encode(prompt, add_special_tokens=False).ids

    bos_id = tokenizer.token_to_id("<bos>")
    if bos_id is not None:
        ids = [bos_id] + ids

    x = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(
        x,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        greedy=False,
    )

    text = tokenizer.decode(out[0].tolist(), skip_special_tokens=True).strip()
    if "### Response:" in text:
        text = text.split("### Response:", 1)[-1].strip()
    if "### Instruction:" in text:
        text = text.split("### Instruction:", 1)[0].strip()
    return text


device = "cuda" if torch.cuda.is_available() else "cpu"
base_path = find_base_checkpoint()
adapter_path = find_adapter_checkpoint(ADAPTER_DIR)

print("Device:", device)
print("Base checkpoint:", base_path)
print("Adapter checkpoint:", adapter_path)

base_state = torch.load(base_path, map_location=device)
adapter_state = torch.load(adapter_path, map_location=device)

config = GPTConfig(**base_state["config"])
model = GPT(config).to(device)
model.load_state_dict(base_state["model_state_dict"])
model = apply_lora(model).to(device)
model.load_state_dict(adapter_state["lora_state_dict"], strict=False)
model.eval()

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

for prompt in TEST_PROMPTS:
    print("\n" + "=" * 80)
    print("PROMPT:", prompt)
    print("\nANSWER:")
    print(answer(model, tokenizer, prompt, device))


## 12. Ask The Model Interactively

Run this after training. It loads existing weights only and defaults to the same `checkpoints/rl` folder used by the RL training cell.


In [ ]:
# Interactive TacticsGPT question cell
# Change ADAPTER_DIR to test another SFT/RL folder.

from pathlib import Path
import glob
import os
import re
import sys
import torch
import torch.nn.functional as F

PROJECT_DIR = Path("/content/drive/MyDrive/TacticsGPT_Phase1_Full_Pretrain")
if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)

SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tokenizers import Tokenizer
from model import GPT, GPTConfig
from train_sft_lora import apply_lora


# Choose which adapter folder to test.
ADAPTER_DIR = "checkpoints/rl"
# ADAPTER_DIR = "checkpoints/sft"

# Optional: set a specific checkpoint file. Leave empty to auto-pick the best/latest.
ADAPTER_CHECKPOINT = ""

TOKENIZER_PATH = "checkpoints/tokenizer/tokenizer.json"
BASE_CHECKPOINT = "checkpoints/pretrain/checkpoint_best.pt"


def latest_ckpt(folder, patterns):
    folder = Path(folder)
    if isinstance(patterns, str):
        patterns = [patterns]

    files = []
    for pattern in patterns:
        files.extend(glob.glob(str(folder / pattern)))

    if not files:
        return None

    def score(path):
        path = Path(path)
        name = path.name
        if name == "checkpoint_best.pt" and path.parent.name == "best_model":
            return 10**18
        if name == "checkpoint_best_reward.pt":
            return 10**18 - 1
        match = re.search(r"(\d+)\.pt$", name)
        return int(match.group(1)) if match else -1

    return max(files, key=score)


def find_base_checkpoint():
    best = Path(BASE_CHECKPOINT)
    if best.exists():
        return str(best)

    ckpt = latest_ckpt("checkpoints/pretrain", "checkpoint_step_*.pt")
    if ckpt:
        return ckpt

    raise FileNotFoundError("No base checkpoint found in checkpoints/pretrain.")


def find_adapter_checkpoint(adapter_dir):
    if ADAPTER_CHECKPOINT:
        path = Path(ADAPTER_CHECKPOINT)
        if not path.exists():
            raise FileNotFoundError(f"ADAPTER_CHECKPOINT does not exist: {path}")
        return str(path)

    adapter_dir = Path(adapter_dir)
    patterns = [
        "best_model/checkpoint_best.pt",
        "checkpoint_best_reward.pt",
        "rl_step_*.pt",
        "sft_step_*.pt",
    ]
    ckpt = latest_ckpt(adapter_dir, patterns)
    if ckpt:
        return ckpt

    raise FileNotFoundError(f"No adapter checkpoint found in {adapter_dir}")


@torch.no_grad()
def generate_answer(question, max_new_tokens=180, temperature=0.7, top_k=40):
    question = question.strip()
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    ids = tokenizer.encode(prompt, add_special_tokens=False).ids

    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")
    if bos_id is not None:
        ids = [bos_id] + ids

    x = torch.tensor([ids], dtype=torch.long, device=device)
    generated = []

    model.eval()
    for _ in range(max_new_tokens):
        x_cond = x[:, -model.config.context_length:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :]

        if temperature <= 0:
            next_id = torch.argmax(logits, dim=-1, keepdim=True)
        else:
            logits = logits / max(temperature, 1e-6)
            if top_k and top_k > 0:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < values[:, [-1]], float("-inf"))
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        token_id = int(next_id.item())
        generated.append(token_id)
        x = torch.cat([x, next_id.view(1, 1)], dim=1)

        if eos_id is not None and token_id == eos_id:
            break

    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    if "### Response:" in text:
        text = text.split("### Response:", 1)[-1].strip()
    if "### Instruction:" in text:
        text = text.split("### Instruction:", 1)[0].strip()
    return text


device = "cuda" if torch.cuda.is_available() else "cpu"
base_path = find_base_checkpoint()
adapter_path = find_adapter_checkpoint(ADAPTER_DIR)

print("Device:", device)
print("Base checkpoint:", base_path)
print("Adapter checkpoint:", adapter_path)

base_state = torch.load(base_path, map_location=device)
config = GPTConfig(**base_state["config"])

model = GPT(config).to(device)
model.load_state_dict(base_state["model_state_dict"])

model = apply_lora(model).to(device)
adapter_state = torch.load(adapter_path, map_location=device)
model.load_state_dict(adapter_state["lora_state_dict"], strict=False)
model.eval()

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("\nReady. Type a question, or type q / quit / exit to stop.")
while True:
    question = input("\nAsk TacticsGPT: ").strip()
    if question.lower() in {"q", "quit", "exit", "stop"}:
        print("Stopped.")
        break
    if not question:
        continue

    answer = generate_answer(
        question,
        max_new_tokens=180,
        temperature=0.7,
        top_k=40,
    )
    print("\nAnswer:\n" + answer)
